In [1]:
from gliner import GLiNER
import pandas as pd
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
import re
import ast
from tqdm import tqdm
tqdm.pandas(desc="Processing Rows")
import os
import ctypes
ctypes.CDLL(
    "/home/ravim/.local/lib/python3.14/site-packages/nvidia/cu13/lib/libnvrtc-builtins.so.13.0"
)
print("loaded")

/apps/common/software/Miniforge3/25.11.0-1-jupyter-base/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loaded


In [2]:

def merge_entities(entities, text):
    if not entities:
        return []
    merged = []
    current = entities[0]
    for next_entity in entities[1:]:
        if next_entity['label'] == current['label'] and (next_entity['start'] == current['end'] + 1 or next_entity['start'] == current['end']):
            current['text'] = text[current['start']: next_entity['end']].strip()
            current['end'] = next_entity['end']
        else:
            merged.append(current)
            current = next_entity
    # Append the last entity
    merged.append(current)
    return merged


# model = GLiNER.from_pretrained("numind/NuNerZero")

model = GLiNER.from_pretrained("urchade/gliner_large-v2.1").to("cuda")

/home/ravim/.local/lib/python3.14/site-packages/huggingface_hub/utils/_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 468.31it/s]


In [3]:
df=pd.read_csv("balanced_sample.csv")
df.head(3)


,identifier,sent,status,msgType,senderName,event,responseType,urgency,severity,certainty,headline,description,instruction,areaDesc,date,time,description_clean
0,NWS-130404-681278,2013-12-25T20:05:46.000Z,Actual,Alert,NWS Springfield MO,Extreme Fire Danger,['Prepare'],Expected,Moderate,Likely,Extreme Fire Danger issued December 25 at 2:05PM CST by NWS Springfield MO,THE GFDI DOES NOT ACCOUNT FOR PRECIPITATION THAT OCCURS WITHIN\nTHE HOUR PRECEDING ANY GIVEN FORECAST HOUR BELOW OR ANY SNOW ON\nTHE GROUND AT THE FORECAST HOUR. PLEASE MAKE APPROPRIATE ADJUSTMENTS.(ALL TIME REFERENCES ARE IN CST)\n\nDAY/DATE 24HR INDEX * MID 3AM 6AM 9AM NOON 3PM 6PM 9PM\nWED DEC 25 LOW 2 * L L L L L L L L \nTHU DEC 26 MODERATE 3 * L L L L L M L M \nFRI DEC 27 MODERATE 3 * L L L L L M L L \nSAT DEC 28 MODERATE 6 * L L L L M M M L \nSUN DEC 29 MODERATE 4 * L L M M M M L L \nMON DEC 30 LOW 1 * L L L L L L L L,NaN,Lawrence,2013-12-25,20:05:46,the gfdi does not account for precipitation that occurs within the hour preceding any given forecast hour below or any snow on the ground at the forecast hour. please make appropriate adjustments.(all time references are in cst) day/date 24hr index * mid 3am 6am 9am noon 3pm 6pm 9pm wed dec 25 low 2 * l l l l l l l l thu dec 26 moderate 3 * l l l l l m l m fri dec 27 moderate 3 * l l l l l m l l sat dec 28 moderate 6 * l l l l m m m l sun dec 29 moderate 4 * l l m m m m l l mon dec 30 low 1 * l l l l l l l l
1,NWS-IDP-PROD-4253625,2020-06-04T21:21:00.000Z,Actual,Alert,NWS Cheyenne WY,Special Weather Statement,['Execute'],Expected,Moderate,Observed,Special Weather Statement issued June 4 at 3:21PM MDT by NWS Cheyenne WY,"At 320 PM MDT, Doppler radar was tracking a strong thunderstorm near\nBox Butte Dam, or 22 miles southwest of Hay Springs, moving east at\n45 mph.\n\nPenny size hail and wind gusts up to 50 mph will be possible with\nthis storm.\n\nLocations impacted include...\nBox Butte Campground and Box Butte Dam.","Torrential rainfall is also occurring with this storm, and may lead\nto localized flooding. Do not drive your vehicle through flooded\nroadways.\n\nThis storm may intensify, so be certain to monitor local radio\nstations and available television stations for additional information\nand possible warnings from the National Weather Service.",Dawes; Box Butte,2020-06-04,21:21:00,"at 320 pm mdt, doppler radar was tracking a strong thunderstorm near box butte dam, or 22 miles southwest of hay springs, moving east at 45 mph. penny size hail and wind gusts up to 50 mph will be possible with this storm. locations impacted include... box butte campground and box butte dam."
2,NWS-130404-683048,2013-12-27T09:12:20.000Z,Actual,Alert,NWS Springfield MO,Extreme Fire Danger,['Prepare'],Expected,Moderate,Likely,Extreme Fire Danger issued December 27 at 3:12AM CST by NWS Springfield MO,THE GFDI DOES NOT ACCOUNT FOR PRECIPITATION THAT OCCURS WITHIN\nTHE HOUR PRECEDING ANY GIVEN FORECAST HOUR BELOW OR ANY SNOW ON\nTHE GROUND AT THE FORECAST HOUR. PLEASE MAKE APPROPRIATE ADJUSTMENTS.(ALL TIME REFERENCES ARE IN CST)\n\nDAY/DATE 24HR INDEX * MID 3AM 6AM 9AM NOON 3PM 6PM 9PM\nFRI DEC 27 MODERATE 3 * L L L L L M L L \nSAT DEC 28 MODERATE 6 * L L L M M M M L \nSUN DEC 29 MODERATE 6 * L L M M M M M L \nMON DEC 30 LOW 2 * L L L L L L L L \nTUE DEC 31 LOW 2 * L L L L L L L L \nWED JAN 01 LOW 1 * L L L L L L L L,NaN,St. Clair,2013-12-27,09:12:20,the gfdi does not account for precipitation that occurs within the hour preceding any given forecast hour below or any snow on the ground at the forecast hour. please make appropriate adjustments.(all time references are in cst) day/date 24hr index * mid 3am 6am 9am noon 3pm 6pm 9pm fri dec 27 moderate 3 * l l l l l m l l sat dec 28 moderate 6 * l l l m m m m l sun dec 29 moderate 6 * l l m m m m m l mon dec 30 low 2 * l l l l l l l l tue dec 31 low 2 * l l l l l l l l wed jan 01 low 1 * l l l l l l l l


In [7]:
# df=df[0:3]

In [8]:
# labels = [
#     "warning type",
#     "time of occurrence",
#     "duration of hazard phrase",
#     "frequency of hazard phrase",
#     "location",
#     "hazard intensity",
#     "hazard quantity",
#     "certainty expression",
#     "protective instruction",
#     "impact or consequence"
# ]
# labels = [l.lower() for l in labels]

# text = """
# * TIMING...LIGHT SNOW WILL REDEVELOP BY LATE MORNING AND CONTINUE\n
# THROUGH THE DAY. BLIZZARD CONDITIONS WILL DEVELOP EARLY THIS\n
# AFTERNOON.\n\n* SNOW ACCUMULATIONS...ADDITIONAL 3 TO 6 INCHES.
# \n\n* WINDS...WEST TO NORTHWEST 25 TO 35 MPH WITH GUSTS UP TO 45 MPH.
# \n\n* VISIBILITIES...NEAR ZERO...ESPECIALLY IN OPEN AREAS. DO NOT GO OUTSIDE.
# """
# text = re.sub(r"\n+", " ", text).replace(" * "," ").lower().strip()
# print(text)
# print("\n")
# entities = model.predict_entities(text, labels, threshold=0.2)

# entities = merge_entities(entities, text)

# for entity in entities:
#     print(entity["text"], "=>", entity["label"])


timing...light snow will redevelop by late morning and continue through the day. blizzard conditions will develop early this afternoon. snow accumulations...additional 3 to 6 inches. winds...west to northwest 25 to 35 mph with gusts up to 45 mph. visibilities...near zero...especially in open areas. do not go outside.


timing => time of occurrence
light snow => hazard quantity
late morning => time of occurrence
continue through the day => duration of hazard phrase
blizzard conditions => warning type
early this afternoon => time of occurrence
snow accumulations => hazard quantity
additional 3 to 6 inches => hazard quantity
winds => hazard quantity
25 to 35 mph => hazard intensity
gusts up to 45 mph => hazard intensity
visibilities => hazard quantity
near zero => hazard intensity
especially in open areas => location
do not go outside => protective instruction


In [4]:
hedges = open("hedges.txt", 'r')
hedges = [line.strip("\n") for line in hedges.readlines()]
hedges = [h.strip() for h in hedges if h.strip()]
# hedges[0:1000]

In [5]:
classes = ["Minor", "Moderate", "Severe", "Extreme"]
n = 100  
balanced = (
    df[df["severity"].isin(classes)]
    .groupby("severity", group_keys=False)
    .sample(n=n, random_state=42)
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)
balanced
balanced['severity'].value_counts()

severity
Moderate    100
Extreme     100
Severe      100
Minor       100
Name: count, dtype: int64

In [6]:
labels = [
    "warning type",
    "time of occurrence",
    "duration of hazard phrase",
    "frequency of hazard phrase",
    "location",
    "hazard intensity",
    "hazard quantity",
    "certainty expression",
    "protective instruction",
    "impact or consequence",
    # "forecast"
]
labels = [l.lower() for l in labels]
def annotate(text):
    text = re.sub(r"\n+", " ", text).lower()
    entities = model.predict_entities(text, labels)
    entities = merge_entities(entities, text)
    temp=dict()
    for i in labels:
        temp[i]=[]
    for entity in entities:
        temp[entity["label"]].append(entity["text"])
    temp["certainty expression"].extend(
        [elem for elem in hedges if elem.lower() in text]
    )
    return temp

In [7]:
balanced['info']=balanced['description'].progress_apply(lambda x: annotate(x))

Processing Rows:   1%|▏         | 5/400 [00:05<06:13,  1.06it/s]/home/ravim/.local/lib/python3.14/site-packages/gliner/data_processing/processor.py:422: UserWarning: Sentence of length 967 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
Processing Rows:   2%|▏         | 7/400 [00:05<03:36,  1.81it/s]/home/ravim/.local/lib/python3.14/site-packages/gliner/data_processing/processor.py:422: UserWarning: Sentence of length 573 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
Processing Rows:  10%|█         | 42/400 [00:08<00:31, 11.34it/s]/home/ravim/.local/lib/python3.14/site-packages/gliner/data_processing/processor.py:422: UserWarning: Sentence of length 888 has been truncated to 384
  batch = [self.preprocess_example(b["tokenized_text"], b[key], class_to_ids) for b in batch_list]
Processing Rows:  17%|█▋        | 68/400 [00:11<00:30, 11

In [8]:
df_dict = pd.json_normalize(balanced['info'])
finaldf=pd.concat([balanced.drop(columns=['info']), df_dict], axis=1)
finaldf.to_csv("annotated.csv",index=False)

In [9]:
finaldf

,identifier,sent,status,msgType,senderName,event,responseType,urgency,severity,certainty,headline,description,instruction,areaDesc,date,time,description_clean,warning type,time of occurrence,duration of hazard phrase,frequency of hazard phrase,location,hazard intensity,hazard quantity,certainty expression,protective instruction,impact or consequence
0,urn:oid:2.49.0.1.840.0.b65cf952732647052c047cb0efaafe4bf6703291.005.2,2022-12-22T04:28:00.000Z,Actual,Update,NWS Denver CO,Winter Weather Advisory,['Execute'],Expected,Moderate,Likely,Winter Weather Advisory issued December 21 at 9:28PM MST until December 22 at 9:00AM MST by NWS Denver CO,"* WHAT...Winds gusting as high as 50 mph, snowfall accumulations up\nto 3 inches, dangerously cold wind chills as low as 55 below zero.\n\n* WHERE...The far northeast plains of Colorado, including northeast\nWeld, Morgan, Washington, Logan, Phillips and Sedgwick Counties.\n\n* WHEN...For the Wind Chill Warning, until 11 AM MST Friday. For the\nWinter Weather Advisory, until 9 AM MST Thursday.\n\n* IMPACTS...Plan on slippery roads. Blowing snow could significantly\nreduce visibility. The dangerously cold wind chills could cause\nfrostbite on exposed skin in as little as 5 minutes.\n","The combination of wind and cold temperatures will create\ndangerously low wind chill values. Avoid outside activities if\npossible. When outside, make sure you wear appropriate clothing, a\nhat, and gloves.\n\nSlow down and use caution while traveling. The latest road\nconditions for Colorado can be obtained by calling 5 1 1 or by going\nto www.cotrip.org.",NaN,2022-12-22,04:28:00,"* what...winds gusting as high as 50 mph, snowfall accumulations up to 3 inches, dangerously cold wind chills as low as 55 below zero. * where...the far northeast plains of colorado, including northeast weld, morgan, washington, logan, phillips and sedgwick counties. * when...for the wind chill warning, until 11 am mst friday. for the winter weather advisory, until 9 am mst thursday. * impacts...plan on slippery roads. blowing snow could significantly reduce visibility. the dangerously cold wind chills could cause frostbite on exposed skin in as little as 5 minutes.",[winter weather advisory],[as little as 5 minutes],"[until 11 am mst friday, until 9 am mst thursday]",[],"[the far northeast plains of colorado, northeast weld, morgan, washington, logan, phillips, sedgwick counties]","[as high as 50 mph, dangerously cold wind chills as low as 55 below zero, dangerously cold wind chills]",[up to 3 inches],"[could, little, can]",[plan on slippery roads],"[blowing snow could significantly reduce visibility, could cause frostbite on exposed skin]"
1,NWS-130404-694658-481250,2014-01-03T17:56:29.000Z,Actual,Update,NWS Buffalo NY,Wind Chill Advisory,['Prepare'],Expected,Moderate,Likely,Wind Chill Advisory issued January 3 at 12:55PM EST expiring January 4 at 4:00AM EST by NWS Buffalo NY,* LOCATIONS...ONTARIO AND LIVINGSTON COUNTIES.\n\n* TIMING...WIND CHILL ADVISORY TONIGHT.\n\n* HAZARDS...DANGEROUS LOW WIND CHILLS.\n\n* WINDS...SOUTHWEST 10 TO 15 MPH.\n\n* WIND CHILL VALUES...AS LOW AS 15 BELOW.\n\n* IMPACTS...BITTERLY COLD WIND CHILLS WILL POSE A RISK FOR FROST \n BITE AND HYPOTHERMIA FOR ANYONE OUTSIDE FOR EVEN SHORT PERIODS \n OF TIME.,A WIND CHILL ADVISORY IS ISSUED WHEN STRONG WIND WILL COMBINE\nWITH COLD TEMPERATURES TO CREATE DANGEROUSLY COLD CONDITIONS FOR\nEXPOSED SKIN. THE WIND WILL MAKE IT FEEL LIKE IT IS 15 DEGREES\nBELOW ZERO OR COLDER FOR A PERIOD OF SEVERAL HOURS. IF YOU WILL\nBE OUTDOORS USE COMMON SENSE AND DRESS WARMLY...MAKING SURE THAT\nALL EXPOSED SKIN IS COVERED. IF POSSIBLE...AVOID PROLONGED\nEXPOSURE TO THE COLD TO PREVENT FROSTBITE AND HYPOTHERMIA.\n\n,Livingston; Ontario,2014-01-03,17:56:29,* locations...ontario and livingston counties. * timing...wind chill advisory tonight. * hazards...dangerous low wind chills. * winds...southwest 10 to 15 mph. * wind chill values...as low as 15 below. * impacts...bitterly cold wind